# 0 · Build the approximate multipliers

This notebook is the **one place** to define and (re)generate the approximate
multipliers used by every sweep. Edit the three tables below, run the whole notebook
(*Run → Run All Cells*), and it writes the LUT header files that AdaPT and TransAxx
read. No C, no compiler, no image rebuild.

Each multiplier is built by *recursive composition* (the scheme from the AMPEREISH
paper): four 2-bit **primitives** make a 4-bit **block**, four 4-bit blocks make an
8-bit **multiplier**. So there are three editable tables, from small to large:

1. **`PRIMITIVES`** — the 2-bit truth tables (rarely changed).
2. **`BLOCKS`** — 4-bit blocks: which primitive sits in each quadrant.
3. **`SPECS`** — the 8-bit multipliers: which block sits in each quadrant.

A self-test at the bottom proves the engine still reproduces the original five
`FPGA_ISH` multipliers **byte-for-byte against the C reference**, so you can trust
anything new you build with it.

## How to add a new multiplier

To add `FPGA_ISH_6` (for example): add one line to **`SPECS`** naming the block for
each quadrant (`HH`, `HL`, `LH`, `LL`). If it needs a block that doesn't exist yet,
add that to **`BLOCKS`** first. Then *Run All Cells*. The new `mul8s_fpga_ish_6.h`
appears for both frameworks and is picked up automatically by the sweep notebooks.

> Quadrants: `HH` = high·high bits, `LL` = low·low bits, etc. `HH` carries the most
> weight (it's the high half of both operands), so approximating there costs the most
> accuracy, and `LL` the least.

In [1]:
import re, json
from pathlib import Path

# --- Where things live -------------------------------------------------------
# OUT_DIR : freshly built headers go here. In the docker setup this is bind-mounted
#           into the framework containers' multiplier dirs, so a new multiplier is
#           usable on the next sweep with no rebuild.
# REF_DIR : the C-generated headers baked into the image. The cross-check below
#           compares against these to prove the Python engine matches.
OUT_DIR = Path("/work/luts")
REF_DIR = Path("/workspace/multipliers/luts")

# Fallback when running outside the container (a plain repo checkout).
if not REF_DIR.exists():
    here = Path.cwd().resolve()
    repo = next((p for p in (here, *here.parents) if (p / "multipliers").is_dir()), here)
    REF_DIR = repo / "multipliers" / "luts"
    OUT_DIR = repo / "docker" / "luts"

print("writing headers to :", OUT_DIR)
print("reference (C) dir  :", REF_DIR, "(exists:", REF_DIR.exists(), ")")

writing headers to : /work/luts
reference (C) dir  : /workspace/multipliers/luts (exists: True )


## 1 · Primitives — the 2-bit building blocks

Each primitive is a 4×4 table: `table[a][b]` is the (approximate) product of the
2-bit numbers `a` and `b` (both in 0..3). `m5` is the exact one. `m1`–`m4` are the
paper's approximate variants. You'll rarely touch these.

In [2]:
PRIMITIVES = {
    "m5": [[0,0,0,0],[0,1,2,3],[0,2,4,6],[0,3,6,9]],   # exact 2-bit multiplier
    "m1": [[0,0,0,0],[0,1,2,3],[0,2,4,6],[0,3,6,7]],   # 3x3 -> 7  (ER 1/16, mag 2)
    "m2": [[0,0,0,0],[0,0,2,2],[0,2,4,6],[0,2,6,9]],   # 1x1=0,1x3=2 (ER 3/16, mag 1)
    "m3": [[0,0,0,0],[0,1,2,3],[0,2,4,6],[0,3,6,11]],  # 3x3 -> 11 (ER 1/16, mag 2)
    "m4": [[0,0,0,0],[0,1,2,3],[0,2,4,6],[0,3,6,5]],   # 3x3 -> 5  (ER 1/16, mag 4)
}

## 2 · Blocks — the 4-bit multipliers

A block names which primitive fills each quadrant of a 4-bit multiply. `exact` uses
the exact primitive everywhere. The `block_XXXX` names follow the paper's labelling.

In [3]:
BLOCKS = {
    "exact":      {"HH": "m5", "HL": "m5", "LH": "m5", "LL": "m5"},
    # power-optimized blocks
    "block_1311": {"HH": "m1", "HL": "m3", "LH": "m1", "LL": "m1"},
    "block_1315": {"HH": "m1", "HL": "m3", "LH": "m1", "LL": "m5"},
    "block_4335": {"HH": "m4", "HL": "m3", "LH": "m3", "LL": "m5"},
    # add new blocks here, e.g.  "block_1555": {"HH":"m1","HL":"m5","LH":"m5","LL":"m5"},
}

## 3 · Multipliers — the 8-bit specs

The headline table. Each entry names which **block** fills each quadrant of the
8-bit multiply. These are the multipliers the sweeps actually use. **Add or edit
here.**

In [4]:
SPECS = {
    "exact":      {"HH": "exact",      "HL": "exact",      "LH": "exact",      "LL": "exact"},
    "FPGA_ISH_1": {"HH": "block_1311", "HL": "block_1311", "LH": "block_1311", "LL": "block_1311"},
    "FPGA_ISH_2": {"HH": "block_1315", "HL": "block_1311", "LH": "block_1311", "LL": "block_1311"},
    "FPGA_ISH_3": {"HH": "block_4335", "HL": "block_1315", "LH": "block_1311", "LL": "block_1311"},
    "FPGA_ISH_4": {"HH": "block_4335", "HL": "block_1315", "LH": "block_1315", "LL": "block_1311"},
    "FPGA_ISH_5": {"HH": "block_4335", "HL": "block_1315", "LH": "block_1315", "LL": "block_1315"},
    # add new multipliers here, e.g.
    # "FPGA_ISH_6": {"HH": "block_4335", "HL": "block_4335", "LH": "block_1315", "LL": "block_1315"},
}

## The composition engine — *don't edit below this line*

This is a Python twin of the C pipeline (`multipliers/compose.c`). The cross-check
cell near the end proves it matches the C output exactly, so you can edit the tables
above with confidence. Sign handling is **sign-magnitude**: multiply the magnitudes,
then the result is negative iff exactly one operand is negative.

In [5]:
def mul_4bit(a, b, block):
    """4-bit multiply: split each operand into 2-bit halves, look each partial
    product up in the quadrant's primitive table, and sum with positional weights."""
    a_h, a_l = (a >> 2) & 0x3, a & 0x3
    b_h, b_l = (b >> 2) & 0x3, b & 0x3
    HH = PRIMITIVES[block["HH"]]; HL = PRIMITIVES[block["HL"]]
    LH = PRIMITIVES[block["LH"]]; LL = PRIMITIVES[block["LL"]]
    return (16 * HH[a_h][b_h]
          +  4 * HL[a_h][b_l]
          +  4 * LH[a_l][b_h]
          +  1 * LL[a_l][b_l])

def mul_8bit(a, b, spec):
    """8-bit unsigned multiply: same idea one level up, over four 4-bit blocks."""
    a_h, a_l = (a >> 4) & 0xF, a & 0xF
    b_h, b_l = (b >> 4) & 0xF, b & 0xF
    return (256 * mul_4bit(a_h, b_h, BLOCKS[spec["HH"]])
          +  16 * mul_4bit(a_h, b_l, BLOCKS[spec["HL"]])
          +  16 * mul_4bit(a_l, b_h, BLOCKS[spec["LH"]])
          +   1 * mul_4bit(a_l, b_l, BLOCKS[spec["LL"]]))

def mul_8bit_signed(a, b, spec):
    """Signed (sign-magnitude) 8-bit multiply for int8 operands a, b in [-128, 127]."""
    mag = mul_8bit(abs(a), abs(b), spec)
    return -mag if (a < 0) != (b < 0) else mag

def _as_int8(byte):
    """Reinterpret an unsigned byte index 0..255 as a signed int8 value."""
    return byte - 256 if byte >= 128 else byte

def compose_lut(spec):
    """Build the full 256x256 LUT. Rows/cols are indexed by the unsigned byte 0..255,
    but each byte is reinterpreted as int8 for the stored product (so lut[255][255]
    == (-1)*(-1) == 1), exactly the convention AdaPT/TransAxx expect."""
    return [[mul_8bit_signed(_as_int8(ai), _as_int8(bi), spec) for bi in range(256)]
            for ai in range(256)]

def compose_lut_from_fn(fn):
    """Escape hatch for an *arbitrary* multiplier (not the recursive scheme): pass any
    fn(a, b) taking signed int8 operands and returning the product. e.g. an EvoApprox
    model, or the exact baseline  compose_lut_from_fn(lambda a, b: a * b)."""
    return [[fn(_as_int8(ai), _as_int8(bi)) for bi in range(256)] for ai in range(256)]

## Build every multiplier

Composes one 256×256 table per entry in `SPECS`, keyed by its AdaPT/TransAxx
identifier `mul8s_<name>`. Add arbitrary multipliers via the escape hatch here too.

In [ ]:
luts = {f"mul8s_{name.lower()}": compose_lut(spec) for name, spec in SPECS.items()}

# --- EvoApprox baseline: faithful mul8s_1L2H -------------------------------
# A real-world approximate multiplier to sit next to the FPGA_ISH set. We do NOT
# use AdaPT's shipped mul8s_1L2H.h: it is sign-broken on negative x negative
# (MAE 116, WCE 32384). Instead we build the LUT from the official EvoApprox C
# circuit (vendored + transpiled under evoapprox/, verified byte-for-byte against
# the compiled C). The circuit takes the raw byte and returns the uint16
# two's-complement product, so we mask to the byte and reinterpret as signed int16.
from evoapprox.mul8s_1L2H import mul8s_1L2H as _evo_1l2h

def _evoapprox(circuit):
    def fn(a, b):                              # a, b: signed int8 from compose_lut_from_fn
        r = circuit(a & 0xFF, b & 0xFF)        # circuit wants the unsigned byte
        return r - 65536 if r >= 32768 else r  # uint16 product -> signed int16
    return fn

luts["mul8s_1l2h"] = compose_lut_from_fn(_evoapprox(_evo_1l2h))

# datasheet self-check: faithful 1L2H must hit MAE 53 / WCE 255 (ehw.fit.vutbr.cz)
_l = luts["mul8s_1l2h"]
_err = [abs(_l[a][b] - _as_int8(a) * _as_int8(b)) for a in range(256) for b in range(256)]
_mae, _wce = sum(_err) / len(_err), max(_err)
assert round(_mae) == 53 and _wce == 255, f"1L2H off datasheet: MAE={_mae}, WCE={_wce}"
print(f"mul8s_1l2h (EvoApprox): MAE={_mae:.2f}, WCE={_wce}  [datasheet: 53 / 255]")

# --- EvoApprox low-error baselines: faithful mul8s_1KVA / mul8s_1KVB --------
# Two more genuine EvoApprox 8x8-signed designs, chosen to bracket the FPGA_ISH set
# on relative error (MRE): 1KVB (MRE 0.90%) sits *inside* the FPGA_ISH band
# (0.64-1.33%) -> a matched-relative-error peer; 1KVA (MRE 0.28%) is the
# highest-accuracy *approximate* multiplier in the whole library -> a near-exact
# anchor below the band. Together with 1L2H (MRE 4.41%, high error) the EvoApprox
# baselines now span low/matched/high MRE, so accuracy-vs-MRE can be traced.
# NB: unlike 1L2H, these circuits' C signature is int16 mul8s(int8, int8) -- they
# take SIGNED operands and return the SIGNED product, so they go straight through
# compose_lut_from_fn (which already feeds signed int8) with no byte wrapper.
from evoapprox.mul8s_1KVA import mul8s_1KVA as _evo_1kva
from evoapprox.mul8s_1KVB import mul8s_1KVB as _evo_1kvb

luts["mul8s_1kva"] = compose_lut_from_fn(_evo_1kva)
luts["mul8s_1kvb"] = compose_lut_from_fn(_evo_1kvb)

# datasheet self-check (ehw.fit.vutbr.cz; datasheet prints the floor of the true MAE)
for _name, _exp_mae, _exp_wce, _ds in [("mul8s_1kva", 1.25, 5, "1.2 / 5"),
                                       ("mul8s_1kvb", 4.25, 17, "4.2 / 17")]:
    _ll = luts[_name]
    _e = [abs(_ll[a][b] - _as_int8(a) * _as_int8(b)) for a in range(256) for b in range(256)]
    _m, _w = sum(_e) / len(_e), max(_e)
    assert round(_m, 2) == _exp_mae and _w == _exp_wce, f"{_name} off datasheet: MAE={_m}, WCE={_w}"
    print(f"{_name} (EvoApprox): MAE={_m:.2f}, WCE={_w}  [datasheet: {_ds}]")

# The shipped (broken) 1L2H is rebuilt separately at the end as a validation
# artifact only -- it is NOT a baseline and is kept out of the manifest.

# --- Optional: more arbitrary multipliers via the same escape hatch --------
# luts["mul8s_exact_check"] = compose_lut_from_fn(lambda a, b: a * b)   # true exact

print(f"built {len(luts)} multipliers:")
for name in luts:
    print("  -", name)

## Emit the C header files

The LUT *values* are identical for both frameworks. Only the header wrapper and the
destination filename differ (AdaPT wants `int16_t` + `<stdint.h>`; TransAxx wants a
`__device__ short` with include guards). So we compute once and write both flavors,
just like the C `build_all.sh`.

In [7]:
def _rows(lut):
    return ",\n".join("\t{ " + ", ".join(str(v) for v in row) + " }" for row in lut)

def emit_adapt(lut, name):
    return (f"/* Generated by 0_build_multipliers.ipynb - do not edit by hand. */\n"
            f"/* Multiplier: {name}   Target: adapt */\n\n"
            f"#include <stdint.h>\n\n"
            f"const int16_t lut[256][256] = {{\n{_rows(lut)}\n}};\n")

def emit_transaxx(lut, name):
    guard = f"_{name.upper()}_H"
    return (f"/* Generated by 0_build_multipliers.ipynb - do not edit by hand. */\n"
            f"/* Multiplier: {name}   Target: transaxx */\n\n"
            f"#ifndef {guard}\n#define {guard}\n\n"
            f"const __device__ short lut[256][256] = {{\n{_rows(lut)}\n}};\n\n"
            f"#endif\n")

TARGETS = {"adapt": emit_adapt, "transaxx": emit_transaxx}

for target, emit in TARGETS.items():
    out = OUT_DIR / target
    out.mkdir(parents=True, exist_ok=True)
    for name, lut in luts.items():
        (out / f"{name}.h").write_text(emit(lut, name))
    print(f"wrote {len(luts)} headers to {out}")

wrote 7 headers to /work/luts/adapt
wrote 7 headers to /work/luts/transaxx


## Cross-check against the C ground truth

This is the safety net. For every multiplier that has a C-generated reference header,
compare the Python output value-for-value. The original `exact` + `FPGA_ISH_1..5`
should all say **MATCH**, which proves the engine is faithful. A multiplier you've
just added shows **NEW** (no reference, expected). If you *redefine* an original one,
it shows **REDEFINED** (fine if you meant to).

In [8]:
def load_ref(path):
    nums = re.findall(r"-?\d+", path.read_text().split("{", 1)[1])[:65536]
    nums = list(map(int, nums))
    return [nums[i*256:(i+1)*256] for i in range(256)]

print(f"{'multiplier':22}  adapt        transaxx")
canon_ok = canon_total = 0
for name, lut in luts.items():
    cells_status = []
    for target in ("adapt", "transaxx"):
        ref = REF_DIR / target / f"{name}.h"
        if not ref.exists():
            cells_status.append("NEW")
        elif lut == load_ref(ref):
            cells_status.append("MATCH"); canon_ok += 1; canon_total += 1
        else:
            cells_status.append("REDEFINED"); canon_total += 1
    print(f"{name:22}  {cells_status[0]:11}  {cells_status[1]}")

print()
if canon_total and canon_ok == canon_total:
    print(f"OK - all {canon_ok} reference multipliers reproduce the C ground truth exactly.")
elif canon_total:
    print(f"NOTE - {canon_ok}/{canon_total} reference checks matched; "
          f"REDEFINED entries mean you changed a multiplier that had a C reference.")
else:
    print("No C references found to check against (fresh checkout?).")

multiplier              adapt        transaxx
mul8s_exact             MATCH        MATCH
mul8s_fpga_ish_1        MATCH        MATCH
mul8s_fpga_ish_2        MATCH        MATCH
mul8s_fpga_ish_3        MATCH        MATCH
mul8s_fpga_ish_4        MATCH        MATCH
mul8s_fpga_ish_5        MATCH        MATCH
mul8s_1l2h              NEW          NEW

OK - all 12 reference multipliers reproduce the C ground truth exactly.


## Write the manifest

Records the list of built multipliers so the sweep notebooks can pick them up
automatically. Define a multiplier here once and it shows up in the sweeps.

In [9]:
manifest_path = OUT_DIR / "multipliers.json"
manifest = {"multipliers": sorted(luts.keys())}
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print("wrote", manifest_path)
print(json.dumps(manifest, indent=2))

wrote /work/luts/multipliers.json
{
  "multipliers": [
    "mul8s_1l2h",
    "mul8s_exact",
    "mul8s_fpga_ish_1",
    "mul8s_fpga_ish_2",
    "mul8s_fpga_ish_3",
    "mul8s_fpga_ish_4",
    "mul8s_fpga_ish_5"
  ]
}


## Done

Headers are written and self-checked. Next:

- **Run a sweep** — open `1_run_sweep_adapt` (or a TransAxx sweep); it reads the
  manifest, so any multiplier you added is already in the matrix.
- **Save with outputs** (Ctrl-S) before committing, so the rendered self-check
  travels with the notebook.

## Validation artifact — AdaPT's bundled 1L2H (not a baseline)

AdaPT ships its own `mul8s_1L2H.h` and it's wrong: the sign is flipped on
negative×negative, so its worst-case LUT error is about 600× ours. The real
`mul8s_1l2h` built above comes from the official EvoApprox circuit instead. The cell
below just regenerates the broken one so the faithful-vs-shipped comparison stays
reproducible. It's kept out of the manifest on purpose, so the sweeps don't pick it
up as a real multiplier — it only exists for the faithful-vs-shipped validation comparison.

In [ ]:
# Validation artifact only -- NOT a baseline, NOT in the manifest.
# Rebuilds AdaPT's bundled (sign-broken) 1L2H from the vendored header so the
# faithful-vs-shipped finding stays reproducible. To actually sweep it, add
# "mul8s_1l2h_shipped" to ../luts/multipliers.json by hand (or to the sweep's MULTS).
_shipped_src = Path("evoapprox/mul8s_1L2H_shipped.h").read_text()
_snums = [int(x) for x in re.findall(r"-?\d+", _shipped_src.split("{", 1)[1])][:65536]
_shipped_lut = [_snums[i * 256:(i + 1) * 256] for i in range(256)]

# confirm it really is the broken one (guards against a mix-up with the faithful)
_serr = [abs(_shipped_lut[a][b] - _as_int8(a) * _as_int8(b)) for a in range(256) for b in range(256)]
_smae, _swce = sum(_serr) / len(_serr), max(_serr)
assert round(_smae) == 116 and _swce == 32384, f"shipped 1L2H not as expected: MAE={_smae}, WCE={_swce}"

for _t, _emit in TARGETS.items():
    (OUT_DIR / _t / "mul8s_1l2h_shipped.h").write_text(_emit(_shipped_lut, "mul8s_1l2h_shipped"))
print(f"validation artifact mul8s_1l2h_shipped re-emitted "
      f"(BROKEN: MAE={_smae:.2f}, WCE={_swce}); excluded from manifest.")